# Rubric Agent

This notebook walks through the three-stage rubric pipeline: extracting
requirements from policy text, refining with historical audit findings,
and producing evidence-backed verdicts against a project description.

**IMPORTANT:** an Ollama server must be running locally (`ollama serve`) with the embedding model configured in `config.yaml` (`embeddings.default.model_name`).

## Imports

In [ ]:
from agentic_patterns.core.rubric import (
    PrintRubricBuilderListener,
    PrintRubricEvaluatorListener,
    PrintRubricRefinerListener,
    RubricBuilder,
    RubricEvaluator,
    refine_with_history,
)
from agentic_patterns.core.vectordb import get_vector_db, MultiSourceRetriever
from agentic_patterns.core.vectordb.chunking import chunk_by_paragraphs
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance

## Stage 1 -- Build rubric from policy

The policy is a simplified SOC 2 subset covering access control, encryption, logging, and incident response. The builder ingests it into a vector store, extracts all MUST/SHOULD/MAY requirements, deduplicates them, and returns a versioned rubric with stable item IDs.

In [ ]:
POLICY_TEXT = """\
Access Control Policy
All production systems MUST enforce role-based access control (RBAC).
User accounts MUST be reviewed quarterly and inactive accounts disabled within 30 days.
Privileged access SHOULD require multi-factor authentication at every login.

Encryption Policy
Data at rest MUST be encrypted using AES-256 or equivalent.
Data in transit MUST use TLS 1.2 or higher for all internal and external communications.
Encryption keys SHOULD be rotated annually and MAY be rotated more frequently for sensitive systems.

Logging and Monitoring Policy
All authentication events MUST be logged with timestamp, user ID, and outcome.
Administrative actions MUST be captured in an immutable audit trail.
Anomaly detection SHOULD be enabled on critical systems and alerts reviewed within 24 hours.

Incident Response Policy
A documented incident response plan MUST be maintained and tested annually.
Security incidents MUST be reported to the security team within one hour of detection.
Post-incident reviews SHOULD be completed within five business days.
"""

In [ ]:
policy_index = get_vector_db("rubric_demo_policy")
policy_index.reset()
policy_index.ingest(
    chunk_by_paragraphs(
        POLICY_TEXT, DocumentProvenance(source="soc2_policy"), min_lines=1
    )
)

In [ ]:
builder = RubricBuilder(listener=PrintRubricBuilderListener())
rubric = await builder.build_from_policy(policy_index, rubric_name="soc2_demo")

In [ ]:
print(f"Rubric: {rubric.rubric_id}  ({len(rubric.items)} items)\n")
for item in rubric.items:
    print(f"[{item.requirement_level.value}] {item.title}  (weight={item.weight})")

## Stage 2 -- Refine with historical findings

The audit history contains findings from three past reviews (Q3, Q1, Q4). Most map to existing rubric items and will bump their weight. Three of them — spread across different quarters — flag the same gap: vendor and contractor accounts not being revoked after engagement end. This topic is absent from the policy, so the LLM should surface it as a new rubric item.

In [ ]:
AUDIT_FINDINGS_TEXT = """\
Q3 Audit Finding: Three service accounts with production access had not been reviewed in over six months.
Q3 Audit Finding: Two internal microservices communicated over plain HTTP instead of TLS.
Q3 Audit Finding: Admin console actions were logged but logs lacked immutability guarantees.
Q3 Audit Finding: A third-party vendor retained SSH access to a production database server for 90 days after project completion.

Q1 Audit Finding: Quarterly access review was completed 15 days late for the payments team.
Q1 Audit Finding: Encryption key rotation had not occurred for the analytics database in 18 months.
Q1 Audit Finding: Two contractor accounts had standing access to the payments service with no defined expiry date.

Q4 Audit Finding: Incident response plan existed but had not been tested since initial creation two years ago.
Q4 Audit Finding: Two critical alerts from the anomaly detection system went unacknowledged for 48 hours.
Q4 Audit Finding: No formal offboarding procedure was followed when an external consultant's engagement ended; access was revoked 47 days later.
"""

In [ ]:
history_index = get_vector_db("rubric_demo_history")
history_index.reset()
history_index.ingest(
    chunk_by_paragraphs(
        AUDIT_FINDINGS_TEXT, DocumentProvenance(source="audit_findings"), min_lines=1
    )
)

In [ ]:
rubric_v2 = await refine_with_history(
    rubric, history_index, listener=PrintRubricRefinerListener()
)

In [ ]:
print(f"Rubric {rubric_v2.rubric_id}  ({len(rubric_v2.items)} items)\n")
for item in rubric_v2.items:
    print(f"[{item.requirement_level.value}] {item.title}  (weight={item.weight:.1f})")

## Stage 3 -- Evidence-backed assessment

Project Aurora is the system being evaluated. For each rubric item the evaluator retrieves evidence from all three indexes (policy, history, project) and produces a verdict: PASS, RISK, or FAIL, with a one-line rationale. The vendor access item promoted in Stage 2 should FAIL immediately — Aurora explicitly has no offboarding process.

In [ ]:
PROJECT_DESCRIPTION_TEXT = """\
Project Aurora -- Security Posture Summary

Aurora enforces RBAC via AWS IAM with quarterly access reviews automated through a custom script.
MFA is required for all human users but not for CI/CD service accounts.

All databases use AES-256 encryption at rest. Internal service-to-service traffic uses mTLS.
Encryption key rotation is handled by AWS KMS with a 365-day rotation policy.

Authentication events are logged to CloudWatch with structured JSON entries.
Admin actions are captured but stored in the same mutable log stream as application logs.

Aurora has a documented incident response runbook. It was last tested eight months ago.
Anomaly detection is not currently enabled; the team relies on manual dashboard reviews.

There is no documented process for revoking vendor or contractor access upon engagement termination.
"""

In [ ]:
project_index = get_vector_db("rubric_demo_project")
project_index.reset()
project_index.ingest(
    chunk_by_paragraphs(
        PROJECT_DESCRIPTION_TEXT,
        DocumentProvenance(source="project_aurora"),
        min_lines=1,
    )
)

In [ ]:
retriever = MultiSourceRetriever(
    policy=policy_index,
    history=history_index,
    project=project_index,
)
evaluator = RubricEvaluator(listener=PrintRubricEvaluatorListener())
verdicts = await evaluator.evaluate(rubric_v2, retriever)

In [ ]:
title_by_id = {item.item_id: item.title for item in rubric_v2.items}
by_status = {"FAIL": [], "RISK": [], "PASS": []}
for v in verdicts:
    by_status[v.status.value].append(v)

for label in ("FAIL", "RISK", "PASS"):
    group = by_status[label]
    if not group:
        continue
    for v in group:
        title = title_by_id.get(v.item_id, v.item_id)
        note = v.rationale.split(".")[0].strip()
        if len(note) > 90:
            note = note[:87] + "..."
        print(f"[{label}]  {title}")
        print(f"       {note}")
    print()